In [108]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from datetime import datetime
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import set_config


In [109]:
set_config(transform_output="pandas")

In [110]:
def convert_time(time_str):
    return datetime.strptime(time_str, "%I:%M:%S %p").hour
# str = "6:00:00 PM"
# str = convert_time(str)
# print(str)


     

In [111]:
df = pd.read_csv("pickleball_data.csv")
columns_to_drop = ["Timestamp", "Email Address", "Date"]
df = df.drop(columns = columns_to_drop)
columns_to_rename = {"Day of the Week" : "day","Time (Hourly)" : "time", "Number of Groups (Don't include your own group!)" : "num_groups", "Number of People (Don't include your own people!)" : "num_people",
                     "Temperature" : "temperature", "Wind Level (0=none, 1=light breeze, 2=wind affecting ball, 3=strong gusts)" : "wind_level", "Weather" : "weather"
                          }
df = df.rename(columns = columns_to_rename)

df[["time"]] = df[["time"]].map(convert_time)

# weather_map = {"Sunny" : 0, }

print(df.head())



        day  time  num_groups  num_people  temperature        weather  \
0    Sunday    18           8          35           65          Sunny   
1  Thursday    15           3           7           80  Partly Cloudy   
2  Thursday    17           4          10           80  Partly Cloudy   
3    Friday    16           6          30           70  Partly Cloudy   
4    Friday    17           6          25           76  Partly Cloudy   

   wind_level  
0           1  
1           2  
2           2  
3           0  
4           0  


In [112]:
encoder = OneHotEncoder(drop='first', sparse_output=False)
categorical_variables = ["day", "weather"]
transform = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_variables)
    ],
    remainder = "passthrough",
    verbose_feature_names_out=False
)
model_df = transform.fit_transform(df)
print(model_df.head())

   day_Saturday  day_Sunday  day_Thursday  day_Tuesday  weather_Partly Cloudy  \
0           0.0         1.0           0.0          0.0                    0.0   
1           0.0         0.0           1.0          0.0                    1.0   
2           0.0         0.0           1.0          0.0                    1.0   
3           0.0         0.0           0.0          0.0                    1.0   
4           0.0         0.0           0.0          0.0                    1.0   

   weather_Rain  weather_Sunny  time  num_groups  num_people  temperature  \
0           0.0            1.0    18           8          35           65   
1           0.0            0.0    15           3           7           80   
2           0.0            0.0    17           4          10           80   
3           0.0            0.0    16           6          30           70   
4           0.0            0.0    17           6          25           76   

   wind_level  
0           1  
1           2  
2 

In [138]:
# print((model_df.columns.to_list()))

predictors = model_df.columns.to_list()
predictors.remove("num_groups")
predictors.remove("num_people")
print(predictors)
response = ["num_groups"]
X_train, X_test, y_train, y_test = train_test_split(model_df[predictors], model_df[response], test_size = 0.2, random_state = 42)

model = LinearRegression().fit(X_train, y_train)

y_pred = model.predict(X_test)

model.score(X_test, y_test)
print(mean_squared_error(y_test, y_pred))
print(root_mean_squared_error(y_test,y_pred))

print(list(zip(predictors, list(model.coef_[0]))))



['day_Saturday', 'day_Sunday', 'day_Thursday', 'day_Tuesday', 'weather_Partly Cloudy', 'weather_Rain', 'weather_Sunny', 'time', 'temperature', 'wind_level']
2.009185797981032
1.4174575118785862
[('day_Saturday', np.float64(0.0)), ('day_Sunday', np.float64(3.131943762054682)), ('day_Thursday', np.float64(2.279467889517592)), ('day_Tuesday', np.float64(2.663013492574254)), ('weather_Partly Cloudy', np.float64(1.4540034105225086)), ('weather_Rain', np.float64(0.9109345013633863)), ('weather_Sunny', np.float64(-0.9427087280311497)), ('time', np.float64(0.13938311372674575)), ('temperature', np.float64(-0.11375980217997794)), ('wind_level', np.float64(-0.7996514505247264))]


In [131]:
test_df = pd.DataFrame(
    {
        'day_Saturday': [0.0],
        'day_Sunday' : [0.0],
        'day_Thursday' : [0.0],
        'day_Tuesday' : [1.0],
        'weather_Partly Cloudy' : [0.0],
        'weather_Rain' : [0.0],
        'weather_Sunny' : [0.0],
        'time' : [15],
        'temperature' : [66],
        'wind_level': [1.0]
    }
)
current_y_pred = model.predict(test_df)
print(current_y_pred)

[[5.70035238]]
